# Task 6 — General Overview & Integration
**Course:** Machine Learning & Deep Learning  
**Points:** 10/60  
**School of Artificial Intelligence and Data Science**

---

## Overview
This notebook synthesises Tasks 1–5 into a coherent survey, comparison table, and reflective
conclusion. Results are read from the saved figures in `../results/`.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## Section 1 — Deep Learning Architecture Survey
(600–800 words covering 5 architectures)

---

### 1.1 Recurrent Neural Networks (RNNs) and LSTM

**Purpose:** Model sequential data where the output depends on previous inputs — e.g. time-
series, natural language, speech.

**Key Innovation — Recurrence:** An RNN maintains a *hidden state* `h_t` that acts as a
memory of prior time-steps:
```
  h_t = σ(W_h·h_{t-1} + W_x·x_t + b)
```
This recurrency gives RNNs theoretically unlimited context, but training them with back-
propagation-through-time (BPTT) causes gradients to vanish or explode because the Jacobian
product grows or shrinks exponentially with sequence length.

**LSTM (Hochreiter & Schmidhuber, 1997)** solves this with a **gated cell state** `C_t`:
- **Forget gate:**  `f_t = σ(W_f·[h_{t-1}, x_t] + b_f)` — what to forget from `C_{t-1}`
- **Input gate:**  `i_t = σ(W_i·[h_{t-1}, x_t] + b_i)` — what new info to store
- **Cell update:** `C̃_t = tanh(W_C·[h_{t-1}, x_t] + b_C)`
- **Output gate:** `o_t = σ(W_o·[h_{t-1}, x_t] + b_o)`,  `h_t = o_t ⊙ tanh(C_t)`

The additive cell-state update prevents vanishing gradients because gradients can flow
through constant-time path `C_t ← C_{t-1}` without multiplicative shrinkage.

**Real-world application:** Language translation (e.g. Google Translate legacy system), stock
price forecasting, and speech recognition with `CTC` loss.

---

### 1.2 Generative Adversarial Networks (GANs)

**Purpose:** Generate new data samples that are indistinguishable from a target distribution.

**Key Innovation — Adversarial Training:** A generator `G(z)` and a discriminator `D(x)` are
trained simultaneously in a two-player minimax game:
```
  min_G  max_D  V(D,G) = E_{x~p_data}[log D(x)] + E_{z~p_z}[log(1-D(G(z)))]
```
- **D** learns to distinguish real vs. fake samples.
- **G** learns to fool D, gradually producing ever-more-realistic outputs.

**Stability tricks:** gradient penalty (WGAN-GP), mode collapse mitigation (ACGAN),
spectral normalisation, progressive growing (StyleGAN).

**Real-world application:** Fixing low-light photographs (EnlightenGAN), fashion-image
synthesis (DeepFashion), and texture synthesis for video games.

---

### 1.3 Transformers and the Attention Mechanism

**Purpose:** Model long-range dependencies in sequences without recurrence; the backbone of
modern NLP and increasingly vision tasks.

**Key Innovation — Self-Attention:** Each token computes a weighted sum of all other tokens
in the sequence, weights learned through Query/Key/Value (QKV) projections:
```
  Attention(Q,K,V) = softmax(Q K^T / √d_k) · V
```
This is fully parallelisable (unlike RNNs) and captures long-range dependencies in O(d)
rather than O(sequence_length) sequential steps.

**Multi-Head Attention** lets the model simultaneously attend from multiple representation
subspaces. A Transformer block stacks:
> Multi-Head Attention → Add & Norm → Feed-Forward → Add & Norm

**Positional Encoding** injects sequence order into an otherwise order-agnostic architecture.

**Real-world application:** Large Language Models (GPT-4, Llama 2), Bing/ChatGPT, and
image understanding (ViT, DETR).

---

### 1.4 Graph Neural Networks (GNNs)

**Purpose:** Learn on graph-structured data — social networks, molecules, recommendation
systems — where the input is a set of nodes related by edges.

**Key Innovation — Message Passing:** Each node aggregates feature vectors from its
neighbours and updates its own representation:
```
  h_v^{(k)} = σ( W^{(k)} · AGG({ h_u^{(k-1)} | u ∈ N(v) }) )
```
After k layers, each node encodes information from its k-hop neighbourhood — analogous to
a CNN receptive field but on irregular graph topologies.

**Architecture families:** GCN (spectral), GraphSAGE (inductive), GAT (attention-weighted),
GIN (universal approximation).

**Real-world application:** Drug discovery (molecular property prediction with GCN),
social network influence modelling, and fraud detection in transaction graphs.

---

### 1.5 Autoencoders (AE) and Variational Autoencoders (VAE)

**Purpose:** Unsupervised dimensionality reduction and generative modelling.

**Autoencoder** learns an encoder `E(x) → z` and a decoder `D(z) → x̂` that minimise
reconstruction loss ‖x − x̂‖², forcing the bottleneck `z` to capture the most salient
features of the data. It is purely a compression / representation learner; the latent space
is unstructured and cannot be sampled.

**VAE (Kingma & Welling, 2013)** adds a probabilistic twist: the encoder outputs a mean
`μ` and variance `σ²`; a sample `z = μ + σ · ε` (ε ~ N(0,1)) is drawn for each input.
The loss includes a KL-divergence regulariser:
```
  L = E[−log p(x | z)] + KL(q_φ(z|x) || p(z))
```
This forces the latent space to follow a continuous, structured prior, enabling smooth
interpolation and generation by sampling from `p(z) = N(0, I)`.

**Real-world application:** Anomaly detection in network traffic; medical-image
denoising; and generative art (style transfer, face generation with β-VAE).

## Section 2 — Comparative Analysis Across All Tasks

In [2]:
results_dict = {
    'Random Forest':         {'task1_gaming': 0.7911, 'task1_wine': 1.0000,
                              'task3_pca_wine': 0.9259, 'task3_pca_digits': 0.9481},
    'SVM':                   {'task1_gaming': 0.7911, 'task1_wine': 0.9630,
                              'task3_pca_wine': 0.9259, 'task3_pca_digits': 0.9481},
    'MLP (Adam, relu)':      {'task1_gaming': None,   'task1_wine': None,
                              'task4_mlp': 0.8550},
    'Custom CNN (CIFAR-10)': {'task5_cnn': 0.7256},
    'ResNet-50 (frozen)':    {'task5_cnn': 0.1704},
}

metrics = []
for model, vals in results_dict.items():
    row = {'Model': model}
    for k, v in vals.items(): row[k] = v if v is not None else '—'
    metrics.append(row)

for r in metrics:
    print(r)

In [3]:
# ── Build comparison table for the report ───────────────────────────────
rows = [
    ('Task 2 — Gaming', 'Random Forest',      '79.11%',  '—',        '289K',   '—'),
    ('Task 2 — Gaming', 'SVM',                '79.11%',  '—',        '13K',    '—'),
    ('Task 2 — Wine',   'Random Forest',      '100.00%', '—',        '289K',   '—'),
    ('Task 2 — Wine',   'SVM',                '96.30%',  '—',        '13K',    '—'),
    ('Task 3 — Wine',   'RF full (13 feat)',  '100.00%', '—',         '—',     '—'),
    ('Task 3 — Wine',   'RF PCA (10 feat)',   '92.59%',  '76.9% dim','—',      '6.9% acc drop'),
    ('Task 3 — Digits', 'RF full (64 feat)',  '96.85%',  '—',         '—',     '—'),
    ('Task 3 — Digits', 'RF PCA (40 feat)',  '94.81%',  '62.5% dim','—',      '2.1% acc drop'),
    ('Task 4 — MLP',    'Adam, lr=0.001',     '85.50%',  '—',       '13KB',   '~20 epochs'),
    ('Task 5 — CIFAR',  'Custom CNN',         '72.56%',  '—',      '254KB',   '~30 epochs'),
    ('Task 5 — CIFAR',  'ResNet-50 frozen',  '17.04%',  '—',      '96.2MB', '~5 epochs'),
]

columns = ['Task / Dataset', 'Model', 'Accuracy', 'Dim Reduction', 'Size', 'Notes']
df_cmp = pd.DataFrame(rows, columns=columns)

plt.figure(figsize=(max(14, len(columns)*2.5), 5))
ax = plt.subplot(111, frame_on=False)
ax.xaxis.set_visible(False); ax.yaxis.set_visible(False)
tbl = ax.table(cellText=df_cmp.values, colLabels=columns, loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.6)
plt.title('Final Comparison — All Tasks (Tasks 1–5)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T6_final_comparison_table.png'), dpi=150)
plt.close()
print('Saved T6_final_comparison_table.png')

df_cmp

## Section 3 — Conclusion (400–500 words)

---
The most valuable insight gained from this project is that **no single algorithm or
architecture is universally optimal** — model selection is necessarily a joint consideration
of dataset size, feature topology, computational budget, and deployment constraints.
On the small, tabular Wine dataset (178 samples, 13 features), classical models — SVM with
RBF kernel and Random Forest — achieved near-perfect accuracy (≈96–100%), far surpassing
any neural network. This demonstrates that classical gradient-boosted or tree-based ensembles
remain competitive when data is scarce and features are already structured.

Dimensionality reduction taught us a second lesson: **lossy compression is a trade-off, not
a free lunch.** PCA retained much of the information on Digits (64 → 40 features, 2.1%
accuracy drop) but applied a harsher squeeze on Wine (13 → 10 features, 6.9% drop). LDA
outperformed PCA for the multi-class case because it is *supervised* and seeks directions that
maximise between-class separation rather than total variance. The practical takeaway is to try
PCA first for speed, then LDA when labelled data is available.

In deep learning, the MLP experiment showed that **Adam converges fastest** on this tabular
data (plateau at epoch 7) versus RMSProp and SGD, yet all three converged to similar final
accuracies (~85%). The deep architectural lesson was reinforced by the CIFAR-10 CNN
experiment: a custom 3-block CNN reached ≈73% accuracy from scratch, while a frozen ResNet-50
stumbled at ≈17% — confirming that ImageNet pre-trained filters do not transfer to 32×32
inputs without fine-tuning. The correct recipe would have been to fine-tune the top layers at
a learning rate several orders of magnitude below the head-training rate.

**Ethical considerations.** All models were trained on synthetic or publicly shared
datasets, but the Gaming_Academic_Performance dataset genders students and grades them by
lifestyle factors (gaming hours, stress level). A classifier trained on this data could
reinforce harmful stereotypes about which students are "high achievers" based on lifestyle
variables correlated with gender or socio-economic background. Before deploying such a model
in education, one would need: (1) a fairness audit across demographic subgroups, (2)
explainability analysis (SHAP/LIME) to identify which features drive predictions, and
(3) explicit human oversight to prevent algorithmic steering of students' self-perceptions.
For the neural-network models, one must also remember that large models memorise training
data — privacy-preserving mitigations (differential privacy, model watermarking,
adversarial training) should be standard practice in production.

**What I would do differently.** Given more time, I would: (1) fine-tune ResNet-50 with
layer-wise unfreezing to close the accuracy gap, (2) add data augmentation to the CIFAR-10
training pipeline (rotation, cutout, mixup), (3) try SMOTE or class weights for the
imbalanced Gaming dataset class-1, and (4) run a hyperparameter sweep (Optuna) on the MLP
architecture width and depth.